# Lesson 04: ChromaDB Vector Database & Hybrid Retrieval (BM25 + Dense RRF)
> **Goal:** Build a complete modern production-grade retrieval pipeline:
> 1. Set up an in-memory **ChromaDB** vector database.
> 2. Ingest and embed paragraph chunks of `data.txt`.
> 3. Perform **Dense Semantic Search** on user questions from `question.txt`.
> 4. Perform **Sparse Keyword Search** using BM25.
> 5. Fuse and rerank both using **Reciprocal Rank Fusion (RRF)**.
> 6. Understand why real-world enterprise RAG systems rely on **Hybrid Search**.

---

### Modern Hybrid Retrieval Architecture
```mermaid
flowchart TD
    A["Knowledge Corpus (data.txt)"] --> B["Paragraph Chunking (30 Chunks)"]
    B --> C["ChromaDB Ingestion<br/>(HNSW Vector Index)"]
    B --> D["BM25 Ingestion<br/>(Sparse Inverted Index)"]
    
    Q["User Query (question.txt)"] --> E["Dense Vector Search<br/>(ChromaDB)"]
    Q --> F["Sparse Keyword Search<br/>(BM25)"]
    
    C --> E
    D --> F
    
    E -->|"Top-K Semantic Candidates"| G["Reciprocal Rank Fusion (RRF)<br/>Score = 1/(60 + Rank)"]
    F -->|"Top-K Keyword Candidates"| G
    
    G --> H["Final Top-K Reranked Context for LLM"]
```


In [1]:
# Step 1: Imports and environment setup
import os, math, re
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv, find_dotenv
import chromadb
from google import genai

# Load environment variables (.env)
load_dotenv(find_dotenv())


True

In [2]:
# Step 2: Minimal ChromaDB setup & Embedding helper
# Create an in-memory ChromaDB client (instant startup, zero configuration)
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="aura_knowledge")

# Google GenAI SDK client for gemini-embedding-001
genai_client = genai.Client()

def embed(texts: list[str]) -> list[list[float]]:
    """Calls Google GenAI gemini-embedding-001 to produce 768-dimensional vectors."""
    response = genai_client.models.embed_content(
        model="gemini-embedding-001",
        contents=texts,
        config={"output_dimensionality": 768}
    )
    return [e.values for e in response.embeddings]


In [3]:
# Step 3: Load document paragraphs and ingest into ChromaDB
for p in [Path("data.txt"), Path("src/notebooks/data.txt"), Path("../data.txt")]:
    if p.exists():
        data_file = p
        break
else:
    raise FileNotFoundError("Could not find data.txt")

with open(data_file, "r", encoding="utf-8") as f:
    chunks = [p.strip() for p in f.read().split("\n\n") if p.strip()]

# Assign clean IDs and compute embeddings for all 30 chunks
chunk_ids = [f"chunk_{i}" for i in range(len(chunks))]
chunk_embeddings = embed(chunks)

# Store text, IDs, and indexed vectors in ChromaDB (upsert is safe to re-run)
collection.upsert(
    ids=chunk_ids,
    documents=chunks,
    embeddings=chunk_embeddings
)

print(f"Successfully ingested {collection.count()} chunks into ChromaDB collection '{collection.name}'.")


Successfully ingested 30 chunks into ChromaDB collection 'aura_knowledge'.


---
## Why Use ChromaDB? (Index vs Linear Scan)

### Inverted Index (BM25) vs Vector Graph (ChromaDB)
- **BM25 (Sparse Index):** Creates a map from word to document IDs: `sku-9921 -> [chunk_22]`. Very fast for exact words, but blind to synonyms.
- **ChromaDB (Dense Index):** Organizes continuous vectors in an **HNSW (Hierarchical Navigable Small World)** graph index. Vectors are nodes connected to their nearest geometric neighbors across layered highway graphs.

### Time Complexity: Linear Scan vs ChromaDB HNSW

| Retrieval Strategy | Search Algorithm | Search Complexity | Query Speed on 1 Million Chunks |
| :--- | :--- | :--- | :--- |
| **Linear Scan (Lesson 03)** | Computes distance against *every* vector | $\mathcal{O}(N \cdot d)$ | $\approx 1{,}000\text{ms} - 2{,}000\text{ms}$ (Slow) |
| **ChromaDB (HNSW)** | Multi-layer greedy graph walk | $\mathcal{O}(\log N \cdot d)$ | $\approx 2\text{ms} - 5\text{ms}$ (Lightning Fast!) |

> **Key Takeaway:** ChromaDB lets RAG scale to millions of documents with near-instant query responses.


In [4]:
# Step 4: Dense Retrieval using ChromaDB
def search_dense(query: str, top_k: int = 3) -> list[dict]:
    """Embeds the query and queries ChromaDB for the closest semantic vectors."""
    query_vector = embed([query])[0]
    raw = collection.query(query_embeddings=[query_vector], n_results=top_k)
    
    results = []
    for doc_id, text, dist in zip(raw["ids"][0], raw["documents"][0], raw["distances"][0]):
        results.append({
            "id": doc_id,
            "text": text,
            "distance": dist
        })
    return results

# Test Question 1
for p in [Path("question.txt"), Path("src/notebooks/question.txt"), Path("../question.txt")]:
    if p.exists():
        question_file = p
        break

with open(question_file, "r", encoding="utf-8") as f:
    questions = [q.strip() for q in f.readlines() if q.strip()]

sample_query = questions[0]
print(f"Query: \"{sample_query}\"\n")

dense_results = search_dense(sample_query, top_k=3)
print("=== Dense Retrieval Results (ChromaDB) ===")
for item in dense_results:
    print(f"[{item['id']}] (Distance: {item['distance']:.4f})")
    print(f"   \"{item['text']}\"\n")


Query: "What is the catalog part number for the Titan-X9 processor and how much revenue did it generate in its first month?"

=== Dense Retrieval Results (ChromaDB) ===
[chunk_22] (Distance: 0.1715)
   "On April 15, 2024, Aura officially commenced commercial availability of the Titan-X9 neural processor, distributed through global supply chains under catalog part number SKU-9921."

[chunk_6] (Distance: 0.2151)
   "Under Dr. Thorne's supervision, the microarchitecture team commenced engineering on the Titan-X9 neural accelerator, a dedicated microchip engineered to execute inference workloads under a strict 5-watt envelope."

[chunk_12] (Distance: 0.2184)
   "Third-party laboratory evaluation in November 2023 confirmed that early Titan-X9 engineering samples demonstrated a 42% latency reduction compared to rival embedded accelerators during real-time image processing."



In [5]:
# Step 5: Sparse Retrieval using BM25
def tokenize(text: str) -> list[str]:
    return re.findall(r"[\w-]+", text.lower())

tokenized_chunks = [tokenize(c) for c in chunks]
num_chunks = len(chunks)
avg_len = sum(len(d) for d in tokenized_chunks) / num_chunks

doc_freq = {}
for doc in tokenized_chunks:
    for word in set(doc):
        doc_freq[word] = doc_freq.get(word, 0) + 1

idf = {}
for word, freq in doc_freq.items():
    idf[word] = math.log((num_chunks - freq + 0.5) / (freq + 0.5) + 1.0)

def search_bm25(query: str, top_k: int = 3) -> list[dict]:
    """Searches chunks using BM25 keyword matching."""
    query_tokens = tokenize(query)
    scores = []
    
    for idx, doc in enumerate(tokenized_chunks):
        doc_len = len(doc)
        doc_counts = Counter(doc)
        len_norm = 0.25 + 0.75 * (doc_len / avg_len)
        
        doc_score = 0.0
        for token in query_tokens:
            if token in idf:
                tf = doc_counts[token]
                doc_score += idf[token] * (tf * 2.5) / (tf + 1.5 * len_norm)
                
        scores.append({
            "id": f"chunk_{idx}",
            "score": doc_score,
            "text": chunks[idx]
        })
        
    scores.sort(key=lambda item: item["score"], reverse=True)
    return scores[:top_k]

bm25_results = search_bm25(sample_query, top_k=3)
print("=== BM25 Retrieval Results ===")
for item in bm25_results:
    print(f"[{item['id']}] (BM25 Score: {item['score']:.2f})")
    print(f"   \"{item['text']}\"\n")


=== BM25 Retrieval Results ===
[chunk_22] (BM25 Score: 16.12)
   "On April 15, 2024, Aura officially commenced commercial availability of the Titan-X9 neural processor, distributed through global supply chains under catalog part number SKU-9921."

[chunk_23] (BM25 Score: 9.19)
   "Sales bookings for SKU-9921 generated $19.4 million during its first thirty days on the market, propelled by large-scale enterprise rollouts across European automated warehouses."

[chunk_15] (BM25 Score: 5.81)
   "For the complete fiscal year 2023, Aura reported consolidated annual revenue of $482.4 million and achieved an adjusted EBITDA of $64.8 million, beating revised guidance."



---
## Hybrid Retrieval: Reciprocal Rank Fusion (RRF)

### The Problem: Incompatible Score Scales
- **BM25 Scores:** Unbounded positive numbers (e.g. `0.0` to `25.0+`). Higher is better.
- **ChromaDB L2 Distances:** Continuous distance metric (e.g. `0.0` to `2.0`). Smaller is better.
- You cannot simply add or average these numbers together because their scales and meanings are completely different!

### The Solution: Reciprocal Rank Fusion (RRF)
RRF completely ignores raw score numbers. Instead, it looks strictly at the **rank position** ($1^{\text{st}}, 2^{\text{nd}}, 3^{\text{rd}}, \dots$) assigned by each search method:

$$\text{RRF Score}(d) = \sum_{m \in \{\text{Dense}, \text{BM25}\}} \frac{1}{k + \text{rank}_m(d)}$$

Where $k = 60$ is the standard smoothing constant:
- Rank #1 gives: $\frac{1}{60 + 1} = 0.01639$
- A document ranked #1 in **both** Dense and BM25 gets a massive boost: $0.01639 + 0.01639 = 0.03278$.
- Documents discovered by only one method still earn fair credit without breaking the ranking.


In [6]:
# Step 6: Hybrid Search Function (Dense + BM25 combined with RRF)
def search_hybrid(query: str, top_k: int = 3, k_rrf: int = 60) -> list[dict]:
    """Combines Dense and BM25 candidate lists using Reciprocal Rank Fusion."""
    # 1. Retrieve top 10 candidates from Dense (ChromaDB)
    dense_matches = search_dense(query, top_k=10)
    
    # 2. Retrieve top 10 candidates from BM25
    bm25_matches = search_bm25(query, top_k=10)
    
    # 3. Accumulate RRF points based on rank positions
    rrf_scores = {}
    text_lookup = {}
    
    for rank, item in enumerate(dense_matches, start=1):
        doc_id = item["id"]
        text_lookup[doc_id] = item["text"]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_rrf + rank))
        
    for rank, item in enumerate(bm25_matches, start=1):
        doc_id = item["id"]
        text_lookup[doc_id] = item["text"]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_rrf + rank))
        
    # 4. Sort documents by total RRF score
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    results = []
    for doc_id, score in sorted_docs[:top_k]:
        results.append({
            "id": doc_id,
            "rrf_score": score,
            "text": text_lookup[doc_id]
        })
    return results

hybrid_results = search_hybrid(sample_query, top_k=3)
print("=== Top 3 Hybrid Reranked Results (Dense + BM25) ===")
for item in hybrid_results:
    print(f"[{item['id']}] (RRF Score: {item['rrf_score']:.5f})")
    print(f"   \"{item['text']}\"\n")


=== Top 3 Hybrid Reranked Results (Dense + BM25) ===
[chunk_22] (RRF Score: 0.03279)
   "On April 15, 2024, Aura officially commenced commercial availability of the Titan-X9 neural processor, distributed through global supply chains under catalog part number SKU-9921."

[chunk_23] (RRF Score: 0.03151)
   "Sales bookings for SKU-9921 generated $19.4 million during its first thirty days on the market, propelled by large-scale enterprise rollouts across European automated warehouses."

[chunk_6] (RRF Score: 0.03128)
   "Under Dr. Thorne's supervision, the microarchitecture team commenced engineering on the Titan-X9 neural accelerator, a dedicated microchip engineered to execute inference workloads under a strict 5-watt envelope."



In [7]:
# Step 7: The Showdown - Comparing Dense, BM25, and Hybrid on 3 contrasting questions
test_questions = [
    # Q1: Exact Part Number / SKU (BM25 & Dense agree)
    questions[0],
    
    # Q2: Overheating issue & hotfix code (BM25 matches keywords; Dense matches symptoms)
    questions[1],
    
    # Q5: Semantic paraphrase: 'power grid connectivity' -> 'off-grid' (Dense & Hybrid excel!)
    questions[4],
]

for q in test_questions:
    print("=" * 80)
    print(f"QUERY: \"{q}\"")
    print("=" * 80)
    
    best_dense = search_dense(q, top_k=1)[0]
    best_bm25 = search_bm25(q, top_k=1)[0]
    best_hybrid = search_hybrid(q, top_k=1)[0]
    
    print(f" Dense Match  : \"{best_dense['text'][:85]}...\"")
    print(f" BM25 Match   : \"{best_bm25['text'][:85]}...\"")
    print(f" Hybrid Match : \"{best_hybrid['text'][:85]}...\"\n")


QUERY: "What is the catalog part number for the Titan-X9 processor and how much revenue did it generate in its first month?"
 Dense Match  : "On April 15, 2024, Aura officially commenced commercial availability of the Titan-X9 ..."
 BM25 Match   : "On April 15, 2024, Aura officially commenced commercial availability of the Titan-X9 ..."
 Hybrid Match : "On April 15, 2024, Aura officially commenced commercial availability of the Titan-X9 ..."

QUERY: "What was the bug incident code logged for the overheating issue and which firmware hotfix version resolved it?"
 Dense Match  : "Shortly after deployment, enterprise customers reported unexpected thermal throttling..."
 BM25 Match   : "Maya Lin’s reliability engineering team diagnosed the root cause within forty-eight h..."
 Hybrid Match : "Shortly after deployment, enterprise customers reported unexpected thermal throttling..."

QUERY: "What technology did the company develop to allow edge devices to function continuously without regular 

---
## Step 8: Augmented Generation (The 'G' in RAG)

Now that we retrieved the top relevant chunks using Hybrid Search, we **augment** the LLM prompt with this factual context to generate an accurate, grounded answer.

```mermaid
flowchart LR
    Q["User Query"] --> H["Hybrid Retriever"]
    H --> C["Top Retrieved Chunks"]
    Q & C --> P["Augmented Prompt<br/>(Context + Question)"]
    P --> LLM["Gemini 3.8 Flash"]
    LLM --> A["Grounded Factual Answer"]
```


In [8]:
# Step 8: Augmented Generation - Generate an answer using the retrieved context
test_query = questions[0]
print(f"User Query: \"{test_query}\"\n")

# 1. Retrieve the top 2 matching chunks using Hybrid Search
retrieved_results = search_hybrid(test_query, top_k=2)
context = "\n\n".join([f"- {item['text']}" for item in retrieved_results])

# 2. Build the augmented prompt
augmented_prompt = f"""You are a helpful assistant. Answer the user question based strictly on the retrieved context below.

Context:
{context}

Question:
{test_query}

Answer:"""

# 3. Call the LLM to generate the grounded answer
gen_response = genai_client.models.generate_content(
    model="gemini-3.8-flash",
    contents=augmented_prompt
)

generated_answer = gen_response.text.strip()
print("=== Generated Answer (LLM + RAG Context) ===")
print(generated_answer)


User Query: "What is the catalog part number for the Titan-X9 processor and how much revenue did it generate in its first month?"

=== Generated Answer (LLM + RAG Context) ===
Based on the provided context, the catalog part number for the Titan-X9 processor is **SKU-9921**, and it generated **$19.4 million** in sales bookings during its first thirty days (first month) on the market.


---
## Step 9: LLM-as-a-Judge (Evaluating Correctness)

In modern automated evaluation pipelines, we verify RAG correctness by having an **LLM Judge** compare the generated answer against the ground truth reference from `answer.txt`.


In [9]:
# Step 9: LLM-as-a-Judge - Evaluate answer correctness against ground truth
from pathlib import Path

# Locate answer.txt
for p in [Path("answer.txt"), Path("src/notebooks/answer.txt"), Path("../answer.txt")]:
    if p.exists():
        answer_file = p
        break
else:
    raise FileNotFoundError("Could not find answer.txt")

with open(answer_file, "r", encoding="utf-8") as f:
    ground_truth_answers = [line.strip() for line in f if line.strip()]

ground_truth = ground_truth_answers[0]  # Reference ground truth for Question 1

# Build the evaluation judge prompt
judge_prompt = f"""You are an expert impartial evaluator grading a RAG system response.

Question:
{test_query}

Ground Truth Answer:
{ground_truth}

Generated Answer:
{generated_answer}

Grade the generated answer for factual correctness against the ground truth.
Respond in this exact format:
VERDICT: [CORRECT or INCORRECT]
EXPLANATION: [1-2 sentences explaining why]"""

judge_response = genai_client.models.generate_content(
    model="gemini-3.8-flash",
    contents=judge_prompt
)

print("=== LLM Judge Evaluation ===")
print(f"Ground Truth : {ground_truth}\n")
print(judge_response.text.strip())


=== LLM Judge Evaluation ===
Ground Truth : The catalog part number is SKU-9921, and it generated $19.4 million during its first thirty days (first month) on the market.

VERDICT: CORRECT
EXPLANATION: The generated answer accurately provides the correct part number (SKU-9921) and the correct revenue figure ($19.4 million) for the first month, fully matching the ground truth.


---
## Summary: The Complete End-to-End RAG Lifecycle

| Stage | Purpose | Technology Used |
| :--- | :--- | :--- |
| **1. Indexing & Storage** | Index paragraphs for sub-linear search | **ChromaDB (HNSW Vector Index)** |
| **2. Sparse Retrieval** | Exact keyword precision (IDs, SKUs, error codes) | **BM25 Inverted Index** |
| **3. Dense Retrieval** | Semantic conceptual understanding | **Gemini Embeddings (`gemini-embedding-001`)** |
| **4. Hybrid Reranking** | Best-of-both-worlds candidate fusion | **Reciprocal Rank Fusion (RRF)** |
| **5. Augmented Generation** | Answer question strictly using retrieved context | **Gemini 3.8 Flash (`generate_content`)** |
| **6. Automated Evaluation** | Verify factual correctness against ground truth | **LLM-as-a-Judge Evaluation** |

### Final Takeaway
You have built a complete, production-grade RAG pipeline: from **raw text chunking** to **hybrid vector retrieval**, through **prompt augmentation**, **grounded LLM generation**, and **automated LLM evaluation**!
